In [2]:
import os
from dotenv import load_dotenv
load_dotenv()
api_key = os.environ.get("GEMINI_API_KEY")

In [3]:
from langchain_community.document_loaders import ArxivLoader

base_docs = ArxivLoader(query="Retrieval Augmented Generation",load_max_docs=5).load()
len(base_docs)

3

In [4]:
for doc in base_docs:
    print(doc.metadata)

{'Published': '2025-06-14', 'Title': 'AR-RAG: Autoregressive Retrieval Augmentation for Image Generation', 'Authors': 'Jingyuan Qi, Zhiyang Xu, Qifan Wang, Lifu Huang', 'Summary': 'We introduce Autoregressive Retrieval Augmentation (AR-RAG), a novel paradigm that enhances image generation by autoregressively incorporating knearest neighbor retrievals at the patch level. Unlike prior methods that perform a single, static retrieval before generation and condition the entire generation on fixed reference images, AR-RAG performs context-aware retrievals at each generation step, using prior-generated patches as queries to retrieve and incorporate the most relevant patch-level visual references, enabling the model to respond to evolving generation needs while avoiding limitations (e.g., over-copying, stylistic bias, etc.) prevalent in existing methods. To realize AR-RAG, we propose two parallel frameworks: (1) Distribution-Augmentation in Decoding (DAiD), a training-free plug-and-use decodin

In [5]:
from langchain_community.vectorstores import Chroma
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
import time

text_splitter = RecursiveCharacterTextSplitter(chunk_size = 500)
split_docs = text_splitter.split_documents(base_docs)

class RateLimitedEmbeddings(GoogleGenerativeAIEmbeddings):
    def embed_documents(self, texts, batch_size=5):  # smaller batch
        all_embeddings = []
        for i in range(0, len(texts), batch_size):
            batch = texts[i:i + batch_size]
            while True:
                try:
                    embeddings = super().embed_documents(batch)
                    all_embeddings.extend(embeddings)
                    time.sleep(1)  # 1s delay between batches
                    break
                except Exception as e:
                    if "429" in str(e) or "RESOURCE_EXHAUSTED" in str(e):
                        print(f"Rate limit hit, waiting 60s... (batch {i//batch_size + 1})")
                        time.sleep(60)  # wait 60s then retry
                    else:
                        raise e
        return all_embeddings
vectorstore = Chroma.from_documents(
    documents=split_docs,
    embedding=RateLimitedEmbeddings(
        model="models/gemini-embedding-001",
        api_key=api_key
    )
)

Rate limit hit, waiting 60s... (batch 22)
Rate limit hit, waiting 60s... (batch 53)


In [6]:
len(split_docs)

382

In [7]:
print(max([len(chunk.page_content) for chunk in split_docs]))

500


In [8]:
retriever = vectorstore.as_retriever(search_kwargs={"k":2})

Creating ground truth dataset beforehand


In [31]:
from langchain_core.prompts import ChatPromptTemplate

template = """Answer the question based only on the following context. If you cannot answer the question with the context, please respond with 'I don't know':

### CONTEXT
{context}

### QUESTION
Question: {question}
"""

prompt = ChatPromptTemplate.from_template(template)

In [32]:
from operator import itemgetter
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableLambda, RunnablePassthrough

primary_qa_llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash",google_api_key=api_key,temperature=0)

retrieval_augmented_qa_chain = (
    # INVOKE CHAIN WITH: {"question" : "<<SOME USER QUESTION>>"}
    # "question" : populated by getting the value of the "question" key
    # "context"  : populated by getting the value of the "question" key and chaining it into the base_retriever
    {"context": itemgetter("question") | retriever, "question": itemgetter("question")}
    # "context"  : is assigned to a RunnablePassthrough object (will not be called or considered in the next step)
    #              by getting the value of the "context" key from the previous step
    | RunnablePassthrough.assign(context=itemgetter("context"))
    # "response" : the "context" and "question" values are used to format our prompt object and then piped
    #              into the LLM and stored in a key called "response"
    # "context"  : populated by getting the value of the "context" key from the previous step
    | {"response": prompt | primary_qa_llm, "context": itemgetter("context")}
)

In [9]:
from langchain_classic.output_parsers.structured import ResponseSchema,StructuredOutputParser

question_schema = ResponseSchema(
    name = "question",
    description = "a description about the context."
)

question_responses_schema = [question_schema,]

question_output_parser = StructuredOutputParser.from_response_schemas(question_responses_schema)
format_instruction = question_output_parser.get_format_instructions()


In [10]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate
question_generation_llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash",google_api_key=api_key,temperature=0)
bare_prompt_template = "{content}"
bare_template = ChatPromptTemplate.from_template(template=bare_prompt_template)

In [11]:
qa_template = """\
You are a University Professor creating a test for advanced students. For each context, create a question that is specific to the context. Avoid creating generic or general questions.

question: a question about the context.

Format the output as JSON with the following keys:
question

context: {context}
"""

prompt_template = ChatPromptTemplate.from_template(template=qa_template)

messages = prompt_template.format_messages(
    context=split_docs[0],
    format_instructions=format_instruction
)

question_generation_chain = bare_template | question_generation_llm

response = question_generation_chain.invoke({"content" : messages})
output_dict = question_output_parser.parse(response.content)

In [12]:
for k, v in output_dict.items():
  print(k)
  print(v)

question
The AR-RAG paper introduces two distinct frameworks, Distribution-Augmentation in Decoding (DAiD) and Feature-Augmentation in Decoding (FAiD), to implement its novel patch-level autoregressive retrieval augmentation. Elaborate on the specific mechanisms by which DAiD and FAiD integrate retrieved visual references into the image generation process, and critically compare their fundamental architectural or methodological differences that allow DAiD to be a 'training-free plug-and-use decoding strategy' while FAiD is a 'parameter-efficient fine-tuning method'.


In [13]:
from tqdm import tqdm
qac_triples = []

for text in tqdm(split_docs[:5]):
    messages = prompt_template.format_messages(
        context = text,
        format_instruction = format_instruction
    )
    response = question_generation_chain.invoke({"content":messages})
    try:
        output_dict = question_output_parser.parse(response.content)
    except Exception as e:
        continue
    output_dict["context"] = text
    qac_triples.append(output_dict)

100%|██████████| 5/5 [01:15<00:00, 15.05s/it]


In [15]:
qac_triples[4]

{'question': "The AR-RAG paper identifies 'over-copying' and 'stylistic bias' as prevalent limitations in existing image generation methods that rely on single, static retrievals. Elaborate on how AR-RAG's core mechanism of performing *context-aware retrievals at each generation step*, specifically by using prior-generated patches as queries, fundamentally mitigates these two particular issues, contrasting its dynamic approach with the static conditioning of prior methods.",
 'context': Document(metadata={'Published': '2025-06-14', 'Title': 'AR-RAG: Autoregressive Retrieval Augmentation for Image Generation', 'Authors': 'Jingyuan Qi, Zhiyang Xu, Qifan Wang, Lifu Huang', 'Summary': 'We introduce Autoregressive Retrieval Augmentation (AR-RAG), a novel paradigm that enhances image generation by autoregressively incorporating knearest neighbor retrievals at the patch level. Unlike prior methods that perform a single, static retrieval before generation and condition the entire generation on

In [16]:
answer_generation_llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash",google_api_key=api_key,temperature=0)

answer_schema = ResponseSchema(
    name = "answer",
    description = "an answer to the question"
)

answer_response_schemas = [answer_schema,]
answer_output_parser = StructuredOutputParser.from_response_schemas(answer_response_schemas)
format_instruction = answer_output_parser.get_format_instructions()

qa_template = """
    You are a University Professor creating a tets for advanced students. For each quetsion and context create an answer.
    answer : an answer about the context
    Format the output as JSON with the following keys:
    answer
    
    question: {question}
    context: {context}
"""

prompt_template = ChatPromptTemplate.from_template(template = qa_template)

messages = prompt_template.format_messages(
    context=qac_triples[0]["context"],
    question=qac_triples[0]["question"],
    format_instructions=format_instruction
)

answer_generation_chain = bare_template | answer_generation_llm

response = answer_generation_chain.invoke({"content" : messages})
output_dict = answer_output_parser.parse(response.content)

In [17]:
for k, v in output_dict.items():
  print(k)
  print(v)

answer
AR-RAG's autoregressive, patch-level retrieval mechanism fundamentally differs from prior methods by performing context-aware retrievals at *each generation step*, using previously generated patches as queries. This contrasts with earlier approaches that rely on a single, static retrieval of entire images before generation, conditioning the entire output on fixed reference images.

This dynamic, patch-level approach mitigates 'over-copying' because the model only retrieves and incorporates the *most relevant patch-level visual references* for the specific part of the image being generated at that moment. Instead of being forced to replicate large portions of a single, pre-selected reference image, AR-RAG can selectively draw fine-grained details from various sources, preventing the wholesale duplication of content. For instance, if a specific texture is needed for a small area, only that texture's patch is retrieved, not an entire image that might contain irrelevant elements.

S

In [ ]:
import time
from tqdm import tqdm

MAX_RETRIES = 5
RETRY_DELAY = 20      # seconds between retries
REQUEST_INTERVAL = 13  # seconds between requests (5 rpm = 1 per 12s, with buffer)

for triple in tqdm(qac_triples):
    messages = prompt_template.format_messages(
        context=triple["context"],
        question=triple["question"],
        format_instructions=format_instruction
    )

    for attempt in range(MAX_RETRIES):
        try:
            response = answer_generation_chain.invoke({"content": messages})
            output_dict = answer_output_parser.parse(response.content)
            triple["answer"] = output_dict["answer"]
            break  # success, exit retry loop

        except Exception as e:
            error_str = str(e)

            if "RESOURCE_EXHAUSTED" in error_str or "429" in error_str:
                # Try to extract the suggested retry delay from the error
                import re
                match = re.search(r'retry in (\d+)', error_str)
                wait = int(match.group(1)) + 5 if match else RETRY_DELAY

                print(f"\n[Rate limited] Waiting {wait}s before retry {attempt + 1}/{MAX_RETRIES}...")
                time.sleep(wait)

            else:
                # Non-rate-limit error — skip this triple
                print(f"\n[Parse error] Skipping triple: {e}")
                break

    time.sleep(REQUEST_INTERVAL)  # Throttle between every request

In [22]:
import pandas as pd
from datasets import Dataset

ground_truth_qac_set = pd.DataFrame(qac_triples)
ground_truth_qac_set["context"] = ground_truth_qac_set["context"].map(lambda x: str(x.page_content))
ground_truth_qac_set = ground_truth_qac_set.rename(columns = {"answer":"ground_truth"})

eval_dataset = Dataset.from_pandas(ground_truth_qac_set)

In [23]:
eval_dataset

Dataset({
    features: ['question', 'context', 'ground_truth'],
    num_rows: 5
})

In [26]:
eval_dataset[0]

{'question': "AR-RAG introduces a novel paradigm that performs context-aware retrievals at each generation step, using prior-generated patches as queries. How does this specific autoregressive, patch-level retrieval mechanism, as opposed to the single, static retrieval of entire images in prior methods, enable AR-RAG to mitigate issues like 'over-copying' and 'stylistic bias' in image generation?",
 'context': 'arXiv:2506.06962v3  [cs.CV]  14 Jun 2025\nAR-RAG: Autoregressive Retrieval Augmentation for\nImage Generation\nJingyuan Qi* 1\nZhiyang Xu* 1\nQifan Wang2\nLifu Huang3\n1Virginia Tech\n2Meta\n3 UC Davis\njingyq1@vt.edu\n(a) Vanilla Image Generation\nPrompt\n(c) Patch-based Autoregressive Retrieval Augmentation (Ours)\n...\nAugmentation\nGeneration\nPrompt\nGenerated Image\nGenerated Image\n(b) Image-Based Retrieval Augmentation\nPrompt\nGenerated Image\nRetrieved Images\n...\nQuery\nKey\nQuery\nKey\nQuery\nValue\nKey\nRetrieval',
 'ground_truth': "AR-RAG mitigates 'over-copying' 

saving the ground truth model as a csv

In [27]:
eval_dataset.to_csv("groundtruth_eval_dataset.csv")

Creating CSV from Arrow format: 100%|██████████| 1/1 [00:00<00:00,  8.47ba/s]


5951

In [39]:
import time
import re
from tqdm import tqdm
from ragas import evaluate

REQUEST_INTERVAL = 60   # seconds between requests
MAX_RETRIES = 5   # more retries

def invoke_with_retry(chain, inputs, request_interval=REQUEST_INTERVAL, max_retries=MAX_RETRIES):
    """Invoke a chain with exponential backoff for rate limiting."""
    for attempt in range(max_retries):
        try:
            result = chain.invoke(inputs)
            time.sleep(request_interval)
            return result
        except Exception as e:
            error_str = str(e)
            if "RESOURCE_EXHAUSTED" in error_str or "429" in error_str:
                # Parse suggested wait from error, else use exponential backoff
                match = re.search(r'retry in (\d+)', error_str)
                if match:
                    wait = int(match.group(1)) + 10  # add buffer on top of suggested
                else:
                    wait = min(30 * (2 ** attempt), 300)  # exponential: 30, 60, 120, 240, 300 max

                print(f"\n[Rate limited] Attempt {attempt+1}/{max_retries} — waiting {wait}s...")
                time.sleep(wait)
            else:
                raise

    raise RuntimeError(f"Failed after {max_retries} retries due to rate limiting.")


def create_ragas_dataset_safe(chain, eval_dataset, request_interval=REQUEST_INTERVAL):
    answers  = []
    contexts = []

    for i, item in enumerate(tqdm(eval_dataset)):
        question = item["question"]
        print(f"\n[{i+1}/{len(eval_dataset)}] Processing: {question[:60]}...")

        try:
            result = invoke_with_retry(chain, {"question": question}, request_interval=request_interval)
            response_text = result["response"].content if hasattr(result["response"], "content") else str(result["response"])
            context_texts = [doc.page_content for doc in result["context"]]
        except RuntimeError as e:
            # If all retries fail, insert empty placeholders and continue
            print(f"[Skipping] Sample {i+1} failed permanently: {e}")
            response_text = ""
            context_texts = []

        answers.append(response_text)
        contexts.append(context_texts)

    ragas_data = {
        "question":     [item["question"] for item in eval_dataset],
        "answer":       answers,
        "contexts":     contexts,
        "ground_truth": [item.get("ground_truth", "") for item in eval_dataset],
    }

    return ragas_data

def evaluate_ragas_safe(ragas_data, metrics, batch_size=1, sleep_between=60):
    """Evaluate one row at a time to avoid rate exhaustion."""
    all_results = []

    rows = [
        {k: ragas_data[k][i] for k in ragas_data}
        for i in range(len(ragas_data["question"]))
    ]

    for i, row in enumerate(tqdm(rows)):
        print(f"\nEvaluating sample {i+1}/{len(rows)}...")
        single_ds = Dataset.from_dict({k: [v] for k, v in row.items()})

        for attempt in range(MAX_RETRIES):
            try:
                result = evaluate(single_ds, metrics=metrics)
                all_results.append(result.to_pandas())
                time.sleep(sleep_between)  # sleep after each row
                break
            except Exception as e:
                error_str = str(e)
                if "RESOURCE_EXHAUSTED" in error_str or "429" in error_str:
                    match = re.search(r'retry in (\d+)', error_str)
                    wait = int(match.group(1)) + 5 if match else 30
                    print(f"[Rate limited] Waiting {wait}s... (attempt {attempt+1}/{MAX_RETRIES})")
                    time.sleep(wait)
                else:
                    print(f"[Error] Skipping sample {i+1}: {e}")
                    break

    return pd.concat(all_results, ignore_index=True)


In [ ]:
from tqdm import tqdm
import pandas as pd

basic_qa_ragas_dataset = create_ragas_dataset_safe(retrieval_augmented_qa_chain, eval_dataset)

In [ ]:
basic_qa_ragas_dataset.to_csv("basic_qa_ragas_dataset.csv")

In [ ]:
basic_qa_result = evaluate_ragas_safe(basic_qa_ragas_dataset)